# Production RAG Deployment

This notebook demonstrates deploying RAG systems to production using the `RAGProvider` class from `docs._technical.providers`.

## What's Covered

1. **Working RAG Pipeline** - Using the `RAGProvider` class
2. **Query Examples** - Testing the pipeline
3. **Caching Demo** - How caching improves performance

---

## Production Concepts (Reference)

For actual production deployment, you would typically use:

- **FastAPI** - To serve RAG as a REST API
- **Docker** - To containerize the application
- **Redis** - For caching responses (advanced)
- **Prometheus** - For monitoring metrics
- **CI/CD** - For automated deployment

See the reference code sections below for examples.

## Working Example: RAG Pipeline

In [1]:
# Import RAG provider directly from file
import sys
import os
import importlib.util

# Get the notebook directory and navigate to project root
notebook_dir = os.path.dirname(os.path.abspath("__FILE__"))
project_root = os.path.dirname(notebook_dir)
providers_path = os.path.join(project_root, "docs", "_technical", "providers.py")

# Load the module directly from the file
spec = importlib.util.spec_from_file_location(
    "providers", 
    providers_path
)
providers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(providers)
RAGProvider = providers.RAGProvider

# Sample documents
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative AI model.",
    "RAG helps reduce hallucinations by grounding responses in retrieved context.",
    "Benefits of RAG include: fresh knowledge, reduced hallucinations, traceability, and cost-effective updates.",
    "Vector search works by converting text into embeddings and finding similar vectors.",
    "Embeddings are numerical representations of text that capture semantic meaning."
]

# Create RAG provider (uses Ollama by default)
rag = RAGProvider(provider="ollama")
rag.add_documents(documents)

print("RAG Pipeline created successfully!")

RAG Pipeline created successfully!


## Query the RAG System

In [2]:
# Query the RAG system
result = rag.query("What is RAG?")

print("Question: What is RAG?")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources retrieved: {len(result['sources'])}")
print(f"Cached: {result['cached']}")

Question: What is RAG?

Answer: RAG stands for Retrieval-Augmented Generation, which is a combination of a retrieval system with a generative AI model.

Sources retrieved: 4
Cached: False


## Caching Demo

The RAGProvider includes built-in caching to improve response times for repeated queries.

In [3]:
# First query - not cached
result1 = rag.query("What are the benefits of RAG?")
print(f"First query - Cached: {result1['cached']}")

# Second query with same question - should be cached
result2 = rag.query("What are the benefits of RAG?")
print(f"Second query - Cached: {result2['cached']}")

# Clear cache
rag.clear_cache()
print("\nCache cleared!")

# Query again - not cached
result3 = rag.query("What are the benefits of RAG?")
print(f"After clear - Cached: {result3['cached']}")

First query - Cached: False
Second query - Cached: True

Cache cleared!
After clear - Cached: False


---

## Reference: FastAPI Server

For production, you'd typically run FastAPI as a separate server. Here's how to integrate `RAGProvider`:

In [ ]:
# Reference: FastAPI Server (run separately, not in notebook)
"""
# main.py - Run this as: uvicorn main:app --reload

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

from docs._technical.providers import RAGProvider

app = FastAPI(title="RAG API")

# Initialize RAG provider
rag = RAGProvider(provider="ollama")

# Add your documents
documents = ["Your documents here..."]
rag.add_documents(documents)

class QueryRequest(BaseModel):
    question: str
    use_cache: bool = True
    top_k: Optional[int] = 4

class QueryResponse(BaseModel):
    answer: str
    sources: List[dict]
    cached: bool = False

@app.post("/query", response_model=QueryResponse)
async def query(request: QueryRequest):
    try:
        result = rag.query(
            question=request.question,
            use_cache=request.use_cache,
            top_k=request.top_k
        )
        return QueryResponse(**result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "healthy"}
"""

print("See code above for FastAPI reference")

## Reference: Docker Deployment

Containerize your RAG application with Docker:

In [ ]:
# Reference: Dockerfile
"""
# Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# Reference: docker-compose.yml
"""
# docker-compose.yml

version: '3.8'

services:
  rag-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OLLAMA_HOST=ollama:11434

  ollama:
    image: ollama/ollama:latest
    ports:
      - "11434:11434"
"""

print("See code above for Docker reference")

## Reference: Environment Configuration

Environment variables for production deployment:

In [ ]:
# Reference: .env.example
"""
# .env.example

# Ollama (local)
OLLAMA_HOST=localhost:11434
OLLAMA_MODEL=llama3.2
OLLAMA_EMBEDDING_MODEL=nomic-embed-text

# Or OpenAI (cloud)
# OPENAI_API_KEY=sk-...
# OPENAI_MODEL=gpt-4o

# Application
LOG_LEVEL=INFO
CACHE_TTL=3600
"""

print("See code above for environment configuration reference")

## Reference: Testing

Write tests for your RAG pipeline:

In [ ]:
# Reference: Testing
"""
# test_rag.py

import pytest
from docs._technical.providers import RAGProvider

@pytest.fixture
def rag():
    provider = RAGProvider(provider="ollama")
    provider.add_documents(["Test document 1", "Test document 2"])
    return provider

def test_rag_query(rag):
    result = rag.query("What is RAG?")
    assert "answer" in result
    assert "sources" in result
    assert len(result["answer"]) > 0

def test_empty_query(rag):
    with pytest.raises(ValueError):
        rag.query("")

def test_caching(rag):
    query = "What is RAG?"
    result1 = rag.query(query, use_cache=True)
    result2 = rag.query(query, use_cache=True)
    assert result1["answer"] == result2["answer"]
    assert result2["cached"] == True
"""

print("See code above for testing reference")

## Next Steps

To deploy to production:

1. Add authentication to FastAPI (e.g., API keys, OAuth)
2. Set up CI/CD pipeline (GitHub Actions, etc.)
3. Configure auto-scaling for high traffic
4. Add detailed logging (structured logs)
5. Set up monitoring with Prometheus and Grafana
6. Consider using Redis for distributed caching